In [ ]:
from snowflake.snowpark.context import get_active_session

from snowflake.snowpark.functions import (
  col,trim,upper,lit,to_date,when,is_null,initcap,
  round as sf_round , sum as sf_sum , count as sf_count,
  year,month,dayofmonth,dayofweek,dayname,monthname,
  quarter,date_part,iff,coalesce
)

from snowflake.snowpark.types import DecimalType
session = get_active_session()
# Target databases/schemas 
session.sql("CREATE OR REPLACE DATABASE PROJECT_MEDALLION").collect()
session.sql("CREATE OR REPLACE SCHEMA PROJECT_MEDALLION.BRONZE").collect()
session.sql("CREATE OR REPLACE SCHEMA PROJECT_MEDALLION.SILVER").collect()
session.sql("CREATE OR REPLACE SCHEMA PROJECT_MEDALLION.GOLD").collect()
SRC="SNOWFLAKE_SAMPLE_DATA.TPCH_SF1"

In [ ]:
session.sql("""
CREATE OR REPLACE FILE FORMAT PROJECT_MEDALLION.BRONZE.CSV_FF
  TYPE = 'CSV'
  FIELD_OPTIONALLY_ENCLOSED_BY = '"'
  FIELD_DELIMITER = ','
  SKIP_HEADER = 1
  NULL_IF = ('', 'NULL')
  """).collect()

In [ ]:
df_customer_raw = session.table(f"{SRC}.CUSTOMER")
df_lineitem_raw = session.table(f"{SRC}.LINEITEM")
df_orders_raw = session.table(f"{SRC}.ORDERS")
df_region_raw = session.table(f"{SRC}.REGION")
df_nation_raw = session.table(f"{SRC}.NATION")

print("Row Counts in Bronze")
for name, df in [ ("CUSTOMER" , df_customer_raw),
    ("LINEITEM",df_lineitem_raw),("ORDERS",df_orders_raw),
    ("REGION",df_region_raw),("NATION",df_nation_raw)
]:
    print(f" {name}: {df.count()}")

In [ ]:
df_customer_raw.write.save_as_table("PROJECT_MEDALLION.BRONZE.CUSTOMER",mode="OVERWRITE")
df_lineitem_raw.write.save_as_table("PROJECT_MEDALLION.BRONZE.LINEITEM",mode="OVERWRITE")
df_orders_raw.write.save_as_table("PROJECT_MEDALLION.BRONZE.ORDERS",mode="OVERWRITE")
df_region_raw.write.save_as_table("PROJECT_MEDALLION.BRONZE.REGION",mode="OVERWRITE")
df_nation_raw.write.save_as_table("PROJECT_MEDALLION.BRONZE.NATION",mode="OVERWRITE")

Now let's clean data -- SILVER LAYER

In [ ]:
df_customer_silver = (
df_customer_raw
.select(
    col("C_CUSTKEY").alias("CUSTOMER_KEY"),
    trim(col("C_NAME")).alias("CUSTOMER_NAME"),
    trim(col("C_ADDRESS")).alias("ADDRESS"),
    col("C_NATIONKEY").alias("NATION_KEY"),
    trim(col("C_PHONE")).alias("PHONE"),
    col("C_ACCTBAL").cast(DecimalType(12,2)).alias("ACCOUNT_BALANCE"),
    upper(trim(col("C_MKTSEGMENT"))).alias("MARKET_SEGMENT"),
   )
	 
    .filter(col("CUSTOMER_KEY").is_not_null())
    .dropDuplicates(["CUSTOMER_KEY"])
    .withColumn(
        "MARKET_SEGMENT",
         coalesce(col("MARKET_SEGMENT"),lit("Unknown"))
       )
)

In [ ]:
df_customer_silver.write.mode("overwrite").save_as_table("PROJECT_MEDALLION.SILVER.CUSTOMER_SILVER")

In [ ]:
df_nation_silver = (
 df_nation_raw
 .select(
  col("N_NATIONKEY").alias("NATION_KEY"),
  initcap(trim(col("N_NAME"))).alias("NAME"),
  col("N_REGIONKEY").alias("REGION_KEY"),
 )
 .dropDuplicates(["NATION_KEY"])
)

df_region_silver = (
 df_region_raw 
 .select(
  col("R_REGIONKEY").alias("REGION_KEY"),
  initcap(trim(col("R_NAME"))).alias("NAME"),
 )
 .dropDuplicates(["REGION_KEY"])
)

In [ ]:
df_nation_silver.write.mode("overwrite").save_as_table("PROJECT_MEDALLION.SILVER.NATION_SILVER")
df_region_silver.write.mode("overwrite").save_as_table("PROJECT_MEDALLION.SILVER.REGION_SILVER")

In [ ]:
df_orders_sliver = (
  df_orders_raw
  .select(
  col("O_ORDERKEY").alias("ORDER_KEY"),
  col("O_CUSTKEY").alias("CUST_KEY"),
  upper(trim(col("O_ORDERSTATUS"))).alias("ORDER_STATUS"),
  col("O_TOTALPRICE").cast(DecimalType(15,2)).alias("TOTAL_PRICE"),
  to_date(col("O_ORDERDATE")).alias("ORDER_DATE"),
  initcap(col("O_ORDERPRIORITY")).alias("ORDER_PRIORITY"),
  trim(col("O_CLERK")).alias("CLERK"), 
  )
  .filter(col("ORDER_KEY").is_not_null())
  .dropDuplicates(["ORDER_KEY"])
  .withColumn(
      "ORDER_STATUS",
      when(col("ORDER_STATUS") == "O", lit("OPEN"))
      .when(col("ORDER_STATUS") == "F", lit("FULLFILLED"))
      .when(col("ORDER_STATUS") == "P", lit("PENDING"))
      .otherwise(col("ORDER_STATUS"))

  )
)

In [ ]:
df_orders_sliver.write.mode("overwrite").save_as_table("PROJECT_MEDALLION.SILVER.ORDERS_SILVER")